# Coral Reef AI — +150 Epochs for BOTH Models

This notebook continues training from the two current best checkpoints:

```text
/content/drive/MyDrive/inferance/best.pt
/content/drive/MyDrive/inferance/best_efficientnetv2s_macro_f1.pth
```

## Training plan

```text
Current YOLO26m best.pt
        ↓
+150 additional epochs
        ↓
Validate OLD vs NEW
        ↓
Promote only if mAP50:95 improves


Current EfficientNetV2-S best checkpoint
        ↓
+150 additional epochs
        ↓
Validation every epoch
        ↓
Save global best by Macro-F1
        ↓
Promote only if Macro-F1 improves
```

So this is **+150 additional epochs for each model**, not “150 total”.

The original checkpoints are never overwritten.

The detector held-out **test split is NOT used**.


In [ ]:
# CELL 1 — INSTALL + GOOGLE DRIVE + GPU

!pip -q install ultralytics==8.4.117 timm==1.0.22 scikit-learn scipy pandas

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from collections import defaultdict
import gc, json, math, random, shutil, time

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode

import timm

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from ultralytics import YOLO
import ultralytics


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError(
        'Enable GPU in Colab: Runtime > Change runtime type > GPU'
    )

DEVICE = torch.device('cuda')
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024**3)

# Colab-safe automatic batches.
if VRAM_GB >= 70:
    YOLO_BATCH, CLS_BATCH = 24, 128
elif VRAM_GB >= 38:
    YOLO_BATCH, CLS_BATCH = 12, 64
elif VRAM_GB >= 22:
    YOLO_BATCH, CLS_BATCH = 8, 32
elif VRAM_GB >= 14:
    YOLO_BATCH, CLS_BATCH = 4, 16
else:
    YOLO_BATCH, CLS_BATCH = 2, 8

print('=' * 110)
print('ENVIRONMENT')
print('=' * 110)
print('GPU          :', GPU_NAME)
print(f'VRAM         : {VRAM_GB:.2f} GB')
print('Torch        :', torch.__version__)
print('Ultralytics  :', ultralytics.__version__)
print('timm         :', timm.__version__)
print('YOLO batch   :', YOLO_BATCH)
print('CLS batch    :', CLS_BATCH)
print('STATUS: ENVIRONMENT_READY')


In [ ]:
# CELL 2 — PATHS + REQUIRED FILES

MYDRIVE = Path('/content/drive/MyDrive')

PROJECT_ROOT = MYDRIVE / 'Coral_Reef_AI_Project'
DET_ROOT = PROJECT_ROOT / '01_Data' / 'Final' / 'Detection_v1'

TRAIN_IMG = DET_ROOT / 'images' / 'train'
VAL_IMG = DET_ROOT / 'images' / 'validation'
TEST_IMG = DET_ROOT / 'images' / 'test'

TRAIN_COCO = DET_ROOT / 'annotations' / 'instances_train.json'
VAL_COCO = DET_ROOT / 'annotations' / 'instances_validation.json'

MANIFEST = (
    PROJECT_ROOT
    / '04_Dataset_Splits'
    / 'classification_roi_manifest_split.csv'
)

INFER_ROOT = MYDRIVE / 'inferance'

OLD_YOLO_BEST = INFER_ROOT / 'best.pt'
OLD_CLS_BEST = INFER_ROOT / 'best_efficientnetv2s_macro_f1.pth'

OUT_ROOT = INFER_ROOT / 'PLUS150_EACH'
YOLO_OUT = OUT_ROOT / 'YOLO26m_PLUS150'
CLS_OUT = OUT_ROOT / 'EfficientNetV2S_PLUS150'

YOLO_OUT.mkdir(parents=True, exist_ok=True)
CLS_OUT.mkdir(parents=True, exist_ok=True)

required = [
    TRAIN_IMG,
    VAL_IMG,
    TRAIN_COCO,
    VAL_COCO,
    MANIFEST,
    OLD_YOLO_BEST,
    OLD_CLS_BEST,
]

for p in required:
    print(('OK   ' if p.exists() else 'MISS '), p)

missing = [p for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        'Missing required files:\n'
        + '\n'.join(str(p) for p in missing)
    )

print()
print('Old YOLO best:', OLD_YOLO_BEST)
print('Old CLS best :', OLD_CLS_BEST)
print('Output       :', OUT_ROOT)
print('STATUS: PATHS_READY')


In [ ]:
# CELL 3 — REBUILD EXACT TRAIN/VALIDATION YOLO DATASET

YOLO_DATA = Path('/content/Coral_YOLO26_PLUS150_Dataset')

if YOLO_DATA.exists():
    shutil.rmtree(YOLO_DATA)

for split in ['train', 'validation']:
    (YOLO_DATA / 'images' / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DATA / 'labels' / split).mkdir(parents=True, exist_ok=True)


def resolve_source(folder, file_name):
    name = Path(file_name).name
    p = folder / name
    if p.exists():
        return p

    matches = list(folder.glob(Path(name).stem + '.*'))
    if not matches:
        raise FileNotFoundError(name)

    return matches[0]


def prep_yolo(coco_path, src_dir, split):
    data = json.loads(coco_path.read_text(encoding='utf-8'))
    by_img = defaultdict(list)

    for a in data['annotations']:
        by_img[int(a['image_id'])].append(a)

    ni = 0
    nbx = 0

    for im in data['images']:
        src = resolve_source(src_dir, im['file_name'])
        dst = YOLO_DATA / 'images' / split / src.name
        shutil.copy2(src, dst)

        W = float(im['width'])
        H = float(im['height'])

        lines = []

        for a in by_img.get(int(im['id']), []):
            x, y, w, h = map(float, a['bbox'])

            xc = (x + w/2) / W
            yc = (y + h/2) / H

            lines.append(
                f'0 {xc:.8f} {yc:.8f} {w/W:.8f} {h/H:.8f}'
            )

            nbx += 1

        (
            YOLO_DATA
            / 'labels'
            / split
            / f'{dst.stem}.txt'
        ).write_text(
            '\n'.join(lines),
            encoding='utf-8'
        )

        ni += 1

    return ni, nbx


tr_i, tr_b = prep_yolo(
    TRAIN_COCO,
    TRAIN_IMG,
    'train'
)

va_i, va_b = prep_yolo(
    VAL_COCO,
    VAL_IMG,
    'validation'
)

assert (tr_i, tr_b, va_i, va_b) == (
    452,
    1540,
    100,
    354,
)

DATA_YAML = YOLO_DATA / 'data.yaml'

DATA_YAML.write_text(
    'path: ' + str(YOLO_DATA) + '\n'
    + 'train: images/train\n'
    + 'val: images/validation\n'
    + 'names:\n'
    + '  0: target_coral\n',
    encoding='utf-8'
)

print('Train:', tr_i, 'images /', tr_b, 'boxes')
print('Val  :', va_i, 'images /', va_b, 'boxes')
print('Test : untouched / not used')
print('STATUS: YOLO_DATA_READY')


In [ ]:
# CELL 4 — BUILD SAME +10% CONTEXT CLASSIFICATION DATA

CONTEXT_PAD = 0.10
CLS_LOCAL = Path('/content/Coral_EfficientNetV2S_PLUS150_Context')

if CLS_LOCAL.exists():
    shutil.rmtree(CLS_LOCAL)

for split in ['train', 'validation']:
    (CLS_LOCAL / split).mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(MANIFEST)

required_cols = {
    'roi_id',
    'image_stem',
    'raw_detection_label',
    'x1_px',
    'y1_px',
    'x2_px',
    'y2_px',
    'split',
}

missing_cols = required_cols - set(manifest.columns)

if missing_cols:
    raise RuntimeError(
        'Missing manifest columns: '
        + str(sorted(missing_cols))
    )


def source_dir(split):
    return {
        'train': TRAIN_IMG,
        'validation': VAL_IMG,
        'test': TEST_IMG,
    }[split]


def resolve_img(folder, stem):
    p = folder / f'{stem}.jpg'

    if p.exists():
        return p

    matches = list(folder.glob(str(stem) + '.*'))

    if not matches:
        raise FileNotFoundError(stem)

    return matches[0]


def context_crop(im, box, pad=0.10):
    W, H = im.size
    x1, y1, x2, y2 = map(float, box)

    bw = max(1.0, x2 - x1)
    bh = max(1.0, y2 - y1)

    px = bw * pad
    py = bh * pad

    return im.crop(
        (
            max(0, int(math.floor(x1 - px))),
            max(0, int(math.floor(y1 - py))),
            min(W, int(math.ceil(x2 + px))),
            min(H, int(math.ceil(y2 + py))),
        )
    )


rows = []

for split in ['train', 'validation']:

    sdf = manifest[
        manifest['split'].astype(str) == split
    ]

    for _, r in sdf.iterrows():

        cls = str(r['raw_detection_label'])

        out_dir = CLS_LOCAL / split / cls
        out_dir.mkdir(parents=True, exist_ok=True)

        src = resolve_img(
            source_dir(split),
            str(r['image_stem'])
        )

        with Image.open(src) as im:
            crop = context_crop(
                im.convert('RGB'),
                (
                    r.x1_px,
                    r.y1_px,
                    r.x2_px,
                    r.y2_px,
                ),
                CONTEXT_PAD,
            )

            out_path = out_dir / f"{r['roi_id']}.jpg"

            crop.save(
                out_path,
                'JPEG',
                quality=95,
                subsampling=0,
            )

        rows.append(
            {
                'split': split,
                'roi_id': str(r['roi_id']),
                'species': cls,
                'crop_path': str(out_path),
            }
        )


crop_manifest = pd.DataFrame(rows)

assert (
    crop_manifest['split'] == 'train'
).sum() == 1540

assert (
    crop_manifest['split'] == 'validation'
).sum() == 354

print('Train ROIs:', (crop_manifest['split'] == 'train').sum())
print('Val ROIs  :', (crop_manifest['split'] == 'validation').sum())
print('Context   : +10%')
print('STATUS: CLS_DATA_READY')


In [ ]:
# CELL 5 — VALIDATE BOTH CURRENT BEST CHECKPOINTS BEFORE +150

# ------------------------------------------------------------
# YOLO current best
# ------------------------------------------------------------

old_yolo = YOLO(str(OLD_YOLO_BEST))

old_yolo_val = old_yolo.val(
    data=str(DATA_YAML),
    split='val',
    imgsz=960,
    batch=YOLO_BATCH,
    conf=0.001,
    iou=0.70,
    max_det=500,
    device=0,
    workers=2,
    plots=False,
    verbose=True,
)

OLD_YOLO_METRICS = {
    'mAP50_95': float(old_yolo_val.box.map),
    'mAP50': float(old_yolo_val.box.map50),
    'mAP75': float(old_yolo_val.box.map75),
    'precision': float(getattr(old_yolo_val.box, 'mp', np.nan)),
    'recall': float(getattr(old_yolo_val.box, 'mr', np.nan)),
}

print('\nOLD YOLO26m:')
for k, v in OLD_YOLO_METRICS.items():
    print(f'{k:12s}: {v:.6f}')


# ------------------------------------------------------------
# EfficientNet current best
# ------------------------------------------------------------

old_cls_ckpt = torch.load(
    OLD_CLS_BEST,
    map_location=DEVICE,
    weights_only=False,
)

MODEL_NAME = old_cls_ckpt.get(
    'model_name',
    'tf_efficientnetv2_s.in21k_ft_in1k'
)

class_to_idx = old_cls_ckpt['class_to_idx']
idx_to_class = {
    int(v): str(k)
    for k, v
    in class_to_idx.items()
}

NCLS = len(class_to_idx)

CLS_SIZE = int(
    old_cls_ckpt.get(
        'input_size',
        300,
    )
)

MEAN = tuple(
    old_cls_ckpt.get(
        'mean',
        (0.5, 0.5, 0.5),
    )
)

STD = tuple(
    old_cls_ckpt.get(
        'std',
        (0.5, 0.5, 0.5),
    )
)


train_tf = transforms.Compose(
    [
        transforms.RandomResizedCrop(
            CLS_SIZE,
            scale=(0.92, 1.0),
            ratio=(0.94, 1.06),
            interpolation=InterpolationMode.BICUBIC,
        ),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomVerticalFlip(0.08),
        transforms.RandomRotation(
            4,
            interpolation=InterpolationMode.BICUBIC,
        ),
        transforms.ColorJitter(
            brightness=0.10,
            contrast=0.10,
            saturation=0.06,
            hue=0.02,
        ),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ]
)

val_tf = transforms.Compose(
    [
        transforms.Resize(
            (CLS_SIZE, CLS_SIZE),
            interpolation=InterpolationMode.BICUBIC,
        ),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ]
)


train_ds = datasets.ImageFolder(
    CLS_LOCAL / 'train',
    transform=train_tf,
)

train_eval_ds = datasets.ImageFolder(
    CLS_LOCAL / 'train',
    transform=val_tf,
)

val_ds = datasets.ImageFolder(
    CLS_LOCAL / 'validation',
    transform=val_tf,
)

assert train_ds.class_to_idx == class_to_idx
assert val_ds.class_to_idx == class_to_idx


counts = np.bincount(
    [y for _, y in train_ds.samples],
    minlength=NCLS,
).astype(np.float32)

w = np.sqrt(
    counts.max()
    / np.maximum(counts, 1)
)

w = w / w.mean()

class_weights = torch.tensor(
    w,
    dtype=torch.float32,
    device=DEVICE,
)


train_loader = DataLoader(
    train_ds,
    batch_size=CLS_BATCH,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
)

train_eval_loader = DataLoader(
    train_eval_ds,
    batch_size=CLS_BATCH,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=CLS_BATCH,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
)


classifier = timm.create_model(
    MODEL_NAME,
    pretrained=False,
    num_classes=NCLS,
).to(DEVICE)

classifier.load_state_dict(
    old_cls_ckpt['model_state_dict']
)


def evaluate_cls(model, loader):

    model.eval()

    yt = []
    yp = []
    probs_all = []

    with torch.no_grad():

        for x, y in loader:

            x = x.to(
                DEVICE,
                non_blocking=True,
            )

            y = y.to(
                DEVICE,
                non_blocking=True,
            )

            with torch.autocast(
                'cuda',
                dtype=torch.float16,
            ):
                logits = model(x)

            probs = torch.softmax(
                logits.float(),
                dim=1,
            )

            pred = probs.argmax(
                dim=1
            )

            yt.extend(
                y.cpu().tolist()
            )

            yp.extend(
                pred.cpu().tolist()
            )

            probs_all.extend(
                probs.cpu().numpy()
            )

    yt = np.asarray(yt)
    yp = np.asarray(yp)
    probs_all = np.asarray(
        probs_all,
        dtype=np.float32,
    )

    metrics = {
        'accuracy': accuracy_score(yt, yp),
        'macro_precision': precision_score(
            yt,
            yp,
            average='macro',
            zero_division=0,
        ),
        'macro_recall': recall_score(
            yt,
            yp,
            average='macro',
            zero_division=0,
        ),
        'macro_f1': f1_score(
            yt,
            yp,
            average='macro',
            zero_division=0,
        ),
        'weighted_f1': f1_score(
            yt,
            yp,
            average='weighted',
            zero_division=0,
        ),
    }

    return (
        metrics,
        yt,
        yp,
        probs_all,
    )


OLD_CLS_METRICS, _, _, _ = evaluate_cls(
    classifier,
    val_loader,
)

print('\nOLD EfficientNetV2-S:')
for k, v in OLD_CLS_METRICS.items():
    print(f'{k:16s}: {v:.6f}')


(OUT_ROOT / 'old_metrics.json').write_text(
    json.dumps(
        {
            'yolo': OLD_YOLO_METRICS,
            'classifier': OLD_CLS_METRICS,
        },
        indent=2,
    ),
    encoding='utf-8',
)

del old_yolo, old_yolo_val
gc.collect()
torch.cuda.empty_cache()

print('STATUS: OLD_BASELINES_VALIDATED')


In [ ]:
# CELL 6 — YOLO26m: +150 ADDITIONAL EPOCHS

YOLO_EXTRA_EPOCHS = 150

YOLO_RUN_NAME = 'FineTune_From_CurrentBest_PLUS150'

yolo_model = YOLO(
    str(
        OLD_YOLO_BEST
    )
)

print('=' * 110)
print('YOLO26m +150')
print('=' * 110)
print('Start:', OLD_YOLO_BEST)
print('Additional epochs:', YOLO_EXTRA_EPOCHS)
print('imgsz:', 960)
print('batch:', YOLO_BATCH)
print('LR0:', 5e-5)

t0 = time.time()

yolo_model.train(
    data=str(DATA_YAML),

    epochs=YOLO_EXTRA_EPOCHS,

    imgsz=960,
    batch=YOLO_BATCH,

    device=0,
    workers=2,

    optimizer='AdamW',

    # Low-LR continuation from an already-trained best checkpoint.
    lr0=5e-5,
    lrf=0.02,
    weight_decay=5e-4,

    cos_lr=True,
    warmup_epochs=0.5,

    # Effectively allow the full requested 150 epochs.
    patience=150,

    save=True,
    save_period=10,

    seed=SEED,
    deterministic=True,

    # Gentler continuation augmentation.
    degrees=2.0,
    translate=0.04,
    scale=0.10,

    fliplr=0.50,
    flipud=0.05,

    hsv_h=0.010,
    hsv_s=0.15,
    hsv_v=0.12,

    mosaic=0.05,
    mixup=0.0,
    close_mosaic=15,

    multi_scale=0.05,

    max_det=500,
    amp=True,

    project=str(YOLO_OUT),
    name=YOLO_RUN_NAME,
    exist_ok=True,

    plots=True,
    verbose=True,
)

YOLO_RUN = YOLO_OUT / YOLO_RUN_NAME

NEW_YOLO_BEST = (
    YOLO_RUN
    / 'weights'
    / 'best.pt'
)

NEW_YOLO_LAST = (
    YOLO_RUN
    / 'weights'
    / 'last.pt'
)

if not NEW_YOLO_BEST.exists():
    raise FileNotFoundError(
        NEW_YOLO_BEST
    )

print(
    f'YOLO training minutes: {(time.time()-t0)/60:.2f}'
)

print('NEW YOLO BEST:', NEW_YOLO_BEST)
print('NEW YOLO LAST:', NEW_YOLO_LAST)
print('STATUS: YOLO_PLUS150_COMPLETE')


In [ ]:
# CELL 7 — VALIDATE NEW YOLO + PROMOTE ONLY IF BETTER

gc.collect()
torch.cuda.empty_cache()

new_yolo = YOLO(
    str(
        NEW_YOLO_BEST
    )
)

new_yolo_val = new_yolo.val(
    data=str(DATA_YAML),
    split='val',
    imgsz=960,
    batch=YOLO_BATCH,
    conf=0.001,
    iou=0.70,
    max_det=500,
    device=0,
    workers=2,
    plots=True,
    verbose=True,
)

NEW_YOLO_METRICS = {
    'mAP50_95': float(new_yolo_val.box.map),
    'mAP50': float(new_yolo_val.box.map50),
    'mAP75': float(new_yolo_val.box.map75),
    'precision': float(getattr(new_yolo_val.box, 'mp', np.nan)),
    'recall': float(getattr(new_yolo_val.box, 'mr', np.nan)),
}

print('=' * 110)
print('YOLO OLD vs +150')
print('=' * 110)

for k in OLD_YOLO_METRICS:

    old_v = OLD_YOLO_METRICS[k]
    new_v = NEW_YOLO_METRICS[k]

    print(
        f'{k:12s}'
        f' | old={old_v:.6f}'
        f' | new={new_v:.6f}'
        f' | delta={new_v-old_v:+.6f}'
    )


PROMOTED_YOLO = (
    INFER_ROOT
    / 'best_yolo26m_plus150.pt'
)

YOLO_IMPROVED = (
    NEW_YOLO_METRICS['mAP50_95']
    > OLD_YOLO_METRICS['mAP50_95']
)

if YOLO_IMPROVED:
    shutil.copy2(
        NEW_YOLO_BEST,
        PROMOTED_YOLO,
    )

    print(
        'PROMOTED YOLO:',
        PROMOTED_YOLO,
    )

else:
    print(
        'Old YOLO remains better by mAP50:95.'
    )

print('STATUS: YOLO_PLUS150_VALIDATED')


In [ ]:
# CELL 8 — BUILD ADAPTIVE HARD-MINING LOADER FOR CLASSIFIER

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.03,
)


def build_hard_loader(model):

    train_m, yt, yp, probs = evaluate_cls(
        model,
        train_eval_loader,
    )

    true_conf = probs[
        np.arange(
            len(yt)
        ),
        yt,
    ]

    sample_weights = np.ones(
        len(yt),
        dtype=np.float32,
    )

    low = true_conf < 0.65
    wrong = yp != yt

    # Moderate hard-example emphasis; capped to avoid
    # making 150 epochs memorize only the difficult samples.
    sample_weights[low] *= 1.18
    sample_weights[wrong] *= 1.32

    soft_classes = {
        class_to_idx[x]
        for x in [
            'Sarcophyton',
            'Sinularia',
            'WavingHand',
        ]
        if x in class_to_idx
    }

    soft_confusion = np.asarray(
        [
            (
                int(t) in soft_classes
                and int(p) in soft_classes
                and int(t) != int(p)
            )
            for t, p
            in zip(
                yt,
                yp,
            )
        ],
        dtype=bool,
    )

    sample_weights[
        soft_confusion
    ] *= 1.20

    sample_weights = np.minimum(
        sample_weights,
        1.8,
    )

    sampler = WeightedRandomSampler(
        torch.tensor(
            sample_weights,
            dtype=torch.double,
        ),
        num_samples=len(
            sample_weights
        ),
        replacement=True,
    )

    loader = DataLoader(
        train_ds,
        batch_size=CLS_BATCH,
        sampler=sampler,
        num_workers=2,
        pin_memory=True,
        persistent_workers=True,
    )

    return (
        loader,
        {
            'train_macro_f1': float(
                train_m['macro_f1']
            ),
            'wrong': int(
                wrong.sum()
            ),
            'low_conf': int(
                low.sum()
            ),
            'soft_confusion': int(
                soft_confusion.sum()
            ),
            'mean_weight': float(
                sample_weights.mean()
            ),
            'max_weight': float(
                sample_weights.max()
            ),
        }
    )


hard_loader, mining_info = build_hard_loader(
    classifier
)

print(
    json.dumps(
        mining_info,
        indent=2,
    )
)

print('STATUS: HARD_LOADER_READY')


In [ ]:
# CELL 9 — EFFICIENTNETV2-S: +150 ADDITIONAL EPOCHS

CLS_EXTRA_EPOCHS = 150

# Reload exact old best before continuation.
old_cls_ckpt = torch.load(
    OLD_CLS_BEST,
    map_location=DEVICE,
    weights_only=False,
)

classifier.load_state_dict(
    old_cls_ckpt['model_state_dict']
)

for p in classifier.parameters():
    p.requires_grad = True


# Global best starts at the OLD checkpoint performance.
best_macro_f1 = float(
    OLD_CLS_METRICS['macro_f1']
)

best_epoch = 0

NEW_CLS_BEST = (
    CLS_OUT
    / 'best_efficientnetv2s_plus150_macro_f1.pth'
)

CLS_LAST = (
    CLS_OUT
    / 'last_efficientnetv2s_plus150.pth'
)

history = []


optimizer = torch.optim.AdamW(
    classifier.parameters(),
    lr=8e-6,
    weight_decay=1e-4,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CLS_EXTRA_EPOCHS,
    eta_min=2e-7,
)

scaler = torch.amp.GradScaler(
    'cuda',
    enabled=True,
)


t0 = time.time()


for epoch in range(
    1,
    CLS_EXTRA_EPOCHS + 1,
):

    # Refresh hard examples every 25 epochs.
    if (
        epoch == 1
        or (
            epoch - 1
        ) % 25 == 0
    ):

        hard_loader, mining_info = build_hard_loader(
            classifier
        )

        print(
            f'\nHard mining refresh @ epoch {epoch}:',
            mining_info,
        )


    # Keep most epochs broad; every third epoch focuses harder.
    active_loader = (
        hard_loader
        if epoch % 3 == 0
        else train_loader
    )


    classifier.train()

    total_loss = 0.0
    seen = 0


    for x, y in active_loader:

        x = x.to(
            DEVICE,
            non_blocking=True,
        )

        y = y.to(
            DEVICE,
            non_blocking=True,
        )

        optimizer.zero_grad(
            set_to_none=True
        )


        with torch.autocast(
            'cuda',
            dtype=torch.float16,
        ):

            logits = classifier(
                x
            )

            loss = criterion(
                logits,
                y,
            )


        scaler.scale(
            loss
        ).backward()

        scaler.unscale_(
            optimizer
        )

        torch.nn.utils.clip_grad_norm_(
            classifier.parameters(),
            5.0,
        )

        scaler.step(
            optimizer
        )

        scaler.update()


        total_loss += (
            float(
                loss.detach().cpu()
            )
            * x.size(
                0
            )
        )

        seen += x.size(
            0
        )


    scheduler.step()


    val_metrics, _, _, _ = evaluate_cls(
        classifier,
        val_loader,
    )


    row = {
        'epoch': epoch,
        'train_mode': (
            'hard'
            if epoch % 3 == 0
            else 'standard'
        ),
        'train_loss': total_loss / max(
            1,
            seen,
        ),
        'lr': optimizer.param_groups[
            0
        ][
            'lr'
        ],
        **{
            f'val_{k}': float(v)
            for k, v
            in val_metrics.items()
        },
    }

    history.append(
        row
    )


    # Save LAST every epoch so Colab work is preserved.
    torch.save(
        {
            'model_state_dict': classifier.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'model_name': MODEL_NAME,
            'class_to_idx': class_to_idx,
            'input_size': CLS_SIZE,
            'context_padding': CONTEXT_PAD,
            'mean': MEAN,
            'std': STD,
            'epoch': epoch,
            'val_metrics': {
                k: float(v)
                for k, v
                in val_metrics.items()
            },
        },
        CLS_LAST,
    )


    if (
        val_metrics['macro_f1']
        > best_macro_f1
    ):

        best_macro_f1 = float(
            val_metrics[
                'macro_f1'
            ]
        )

        best_epoch = epoch


        torch.save(
            {
                'model_state_dict': classifier.state_dict(),
                'model_name': MODEL_NAME,
                'class_to_idx': class_to_idx,
                'input_size': CLS_SIZE,
                'context_padding': CONTEXT_PAD,
                'mean': MEAN,
                'std': STD,
                'stage': 'PLUS150_continuation',
                'epoch': epoch,
                'val_metrics': {
                    k: float(v)
                    for k, v
                    in val_metrics.items()
                },
            },
            NEW_CLS_BEST,
        )


    pd.DataFrame(
        history
    ).to_csv(
        CLS_OUT
        / 'training_history_plus150.csv',
        index=False,
    )


    print(
        f"Epoch {epoch:03d}/{CLS_EXTRA_EPOCHS}"
        f" | mode={row['train_mode']:8s}"
        f" | loss={row['train_loss']:.4f}"
        f" | lr={row['lr']:.2e}"
        f" | Acc={val_metrics['accuracy']:.4f}"
        f" | MacroF1={val_metrics['macro_f1']:.4f}"
        f" | Best={best_macro_f1:.4f}"
    )


print()
print(
    f'Classifier +150 minutes: {(time.time()-t0)/60:.2f}'
)

print('Old MacroF1 :', OLD_CLS_METRICS['macro_f1'])
print('Best MacroF1:', best_macro_f1)
print('Best epoch  :', best_epoch)

print('STATUS: CLASSIFIER_PLUS150_COMPLETE')


In [ ]:
# CELL 10 — FINAL CLASSIFIER VALIDATION + PROMOTE IF BETTER

CLS_IMPROVED = NEW_CLS_BEST.exists()

if CLS_IMPROVED:

    new_cls_ckpt = torch.load(
        NEW_CLS_BEST,
        map_location=DEVICE,
        weights_only=False,
    )

    classifier.load_state_dict(
        new_cls_ckpt[
            'model_state_dict'
        ]
    )

    NEW_CLS_METRICS, cls_true, cls_pred, _ = evaluate_cls(
        classifier,
        val_loader,
    )


    print('=' * 110)
    print('CLASSIFIER OLD vs +150')
    print('=' * 110)

    for k in OLD_CLS_METRICS:

        old_v = OLD_CLS_METRICS[
            k
        ]

        new_v = NEW_CLS_METRICS[
            k
        ]

        print(
            f'{k:16s}'
            f' | old={old_v:.6f}'
            f' | new={new_v:.6f}'
            f' | delta={new_v-old_v:+.6f}'
        )


    report = pd.DataFrame(
        classification_report(
            cls_true,
            cls_pred,
            target_names=[
                idx_to_class[i]
                for i
                in range(
                    NCLS
                )
            ],
            zero_division=0,
            output_dict=True,
        )
    ).T

    report.to_csv(
        CLS_OUT
        / 'validation_classification_report_plus150.csv'
    )

    np.savetxt(
        CLS_OUT
        / 'validation_confusion_matrix_plus150.csv',
        confusion_matrix(
            cls_true,
            cls_pred,
        ),
        fmt='%d',
        delimiter=',',
    )


    PROMOTED_CLS = (
        INFER_ROOT
        / 'best_efficientnetv2s_plus150.pth'
    )

    shutil.copy2(
        NEW_CLS_BEST,
        PROMOTED_CLS,
    )

    print(
        'PROMOTED CLASSIFIER:',
        PROMOTED_CLS,
    )

else:

    NEW_CLS_METRICS = OLD_CLS_METRICS

    print(
        'No +150 epoch beat the original classifier Macro-F1.'
    )

    print(
        'Original classifier remains best:',
        OLD_CLS_BEST,
    )


print('STATUS: CLASSIFIER_PLUS150_VALIDATED')


In [ ]:
# CELL 11 — FINAL OLD-vs-NEW SUMMARY

FINAL_SUMMARY = {
    'training_request': {
        'YOLO26m_additional_epochs': 150,
        'EfficientNetV2S_additional_epochs': 150,
    },
    'YOLO26m': {
        'old_checkpoint': str(
            OLD_YOLO_BEST
        ),
        'old_validation': OLD_YOLO_METRICS,
        'new_candidate_checkpoint': str(
            NEW_YOLO_BEST
        ),
        'new_validation': NEW_YOLO_METRICS,
        'improved': bool(
            YOLO_IMPROVED
        ),
        'promoted_checkpoint': (
            str(
                PROMOTED_YOLO
            )
            if YOLO_IMPROVED
            else str(
                OLD_YOLO_BEST
            )
        ),
    },
    'EfficientNetV2S': {
        'old_checkpoint': str(
            OLD_CLS_BEST
        ),
        'old_validation': OLD_CLS_METRICS,
        'new_validation': NEW_CLS_METRICS,
        'improved': bool(
            CLS_IMPROVED
        ),
        'best_plus150_epoch': int(
            best_epoch
        ),
        'promoted_checkpoint': (
            str(
                INFER_ROOT
                / 'best_efficientnetv2s_plus150.pth'
            )
            if CLS_IMPROVED
            else str(
                OLD_CLS_BEST
            )
        ),
    },
    'detector_test_used': False,
}


FINAL_JSON = (
    OUT_ROOT
    / 'PLUS150_EACH_FINAL_SUMMARY.json'
)

FINAL_JSON.write_text(
    json.dumps(
        FINAL_SUMMARY,
        indent=2,
        ensure_ascii=False,
    ),
    encoding='utf-8',
)


print('=' * 120)
print('BOTH +150 TRAINING COMPLETE')
print('=' * 120)

print(
    'YOLO promoted:',
    FINAL_SUMMARY[
        'YOLO26m'
    ][
        'promoted_checkpoint'
    ]
)

print(
    'Classifier promoted:',
    FINAL_SUMMARY[
        'EfficientNetV2S'
    ][
        'promoted_checkpoint'
    ]
)

print(
    'Summary:',
    FINAL_JSON
)

print(
    'Detector held-out TEST used:',
    False
)

print('STATUS: COMPLETE')


## After the +150 / +150 run

If the extended YOLO improves, its promoted checkpoint will be:

```text
/content/drive/MyDrive/inferance/best_yolo26m_plus150.pt
```

If the extended classifier improves, its promoted checkpoint will be:

```text
/content/drive/MyDrive/inferance/best_efficientnetv2s_plus150.pth
```

Next, rerun the **validation detector-confidence sweep and end-to-end evaluation** using the promoted checkpoints before running the final videos.

Do not use the held-out detector test for tuning.
